# Session 7 · Analysing and Optimising Content Strategies

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-07_make_data.py`

In [1]:
"""
Session 7 · Data helper — create a SYNTHETIC posts dataset for the examples.

The numbers are invented by a seeded random generator (so everyone gets the
same file); they mimic the columns a social-media dashboard exports.
Run this first:  python session-07_make_data.py
Creates: session-07_posts.csv
"""
import csv
import random
from datetime import date, timedelta

rng = random.Random(42)
FORMATS = {"reel": (1.6, 0.050), "carousel": (1.1, 0.045), "image": (0.8, 0.030), "text": (0.5, 0.020)}
TOPICS = ["admissions", "student life", "research", "events", "tips"]
start = date(2026, 9, 1)

rows = []
for i in range(60):
    fmt = rng.choice(list(FORMATS))
    reach_mult, base_eng = FORMATS[fmt]
    day = start + timedelta(days=i)
    hour = rng.choice([9, 13, 17, 20])
    boost = 1.3 if hour in (17, 20) else 1.0           # evenings do better in this fake audience
    reach = int(rng.gauss(2000, 400) * reach_mult * boost)
    engagements = int(reach * base_eng * rng.uniform(0.7, 1.3))
    clicks = int(reach * rng.uniform(0.004, 0.015))
    signups = int(clicks * rng.uniform(0.02, 0.10))
    rows.append({"post_id": f"P{i+1:02}", "date": day.isoformat(), "hour": hour, "format": fmt,
                 "topic": rng.choice(TOPICS), "impressions": int(reach * 1.4), "reach": reach,
                 "engagements": engagements, "clicks": clicks, "signups": signups})

with open("session-07_posts.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)
print(f"Wrote {len(rows)} synthetic posts -> session-07_posts.csv")
print("Columns:", ", ".join(rows[0].keys()))
print("First row:", rows[0])

Wrote 60 synthetic posts -> session-07_posts.csv
Columns: post_id, date, hour, format, topic, impressions, reach, engagements, clicks, signups
First row: {'post_id': 'P01', 'date': '2026-09-01', 'hour': 9, 'format': 'reel', 'topic': 'tips', 'impressions': 4443, 'reach': 3174, 'engagements': 124, 'clicks': 16, 'signups': 1}


### Example · `session-07_metrics.py`

In [2]:
"""
Session 7 · Example 1 — Compute content KPIs from a dashboard export.

Definitions used (state them in every report, because platforms differ):
  engagement rate (ER) = engagements / reach x 100
  click-through rate (CTR) = clicks / impressions x 100
  conversion rate (CVR) = sign-ups / clicks x 100
Uses pandas because real exports are CSV tables.

Run:  python session-07_make_data.py   (once, creates the CSV)
      python session-07_metrics.py
"""
import pandas as pd

df = pd.read_csv("session-07_posts.csv")
df["ER %"] = 100 * df.engagements / df.reach
df["CTR %"] = 100 * df.clicks / df.impressions
df["CVR %"] = 100 * df.signups / df.clicks.where(df.clicks > 0)

print("Overall (totals, not averages of percentages):")
print(f"  posts={len(df)}  reach={df.reach.sum():,}  ER={100*df.engagements.sum()/df.reach.sum():.2f}%  "
      f"CTR={100*df.clicks.sum()/df.impressions.sum():.2f}%  CVR={100*df.signups.sum()/df.clicks.sum():.2f}%")

by_fmt = df.groupby("format").agg(posts=("post_id", "count"), reach=("reach", "mean"),
                                  eng=("engagements", "sum"), r=("reach", "sum"))
by_fmt["ER %"] = 100 * by_fmt.eng / by_fmt.r
print("\nBy format:")
print(by_fmt[["posts", "reach", "ER %"]].round(1).sort_values("ER %", ascending=False).to_string())

Overall (totals, not averages of percentages):
  posts=60  reach=141,418  ER=3.99%  CTR=0.60%  CVR=3.47%

By format:
          posts   reach  ER %
format                       
reel         21  3427.1   4.6
carousel     15  2426.3   4.2
image         9  1826.9   3.2
text         15  1107.5   1.8


### Example · `session-07_best_time.py`

In [3]:
"""
Session 7 · Example 3 — When should we post? Engagement by hour and weekday.

Run:  python session-07_best_time.py   (needs session-07_posts.csv)
"""
import pandas as pd

df = pd.read_csv("session-07_posts.csv", parse_dates=["date"])
df["weekday"] = df.date.dt.day_name().str[:3]
df["ER %"] = 100 * df.engagements / df.reach

by_hour = df.groupby("hour")["ER %"].agg(["count", "mean"]).round(2)
print("Engagement rate by posting hour:")
print(by_hour.to_string())
best = by_hour["mean"].idxmax()
print(f"\nBest hour in this data: {best}:00 (n={by_hour.loc[best, 'count']} posts)")

order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_day = df.groupby("weekday")["reach"].mean().reindex(order).round(0)
print("\nAverage reach by weekday:")
print(by_day.to_string())
print("\nCaution: under 10 posts per group is a hint, not proof. Re-check monthly.")

Engagement rate by posting hour:
      count  mean
hour             
9        14  3.49
13       19  3.53
17       13  3.91
20       14  3.46

Best hour in this data: 17:00 (n=13 posts)

Average reach by weekday:
weekday
Mon    2438.0
Tue    2614.0
Wed    2102.0
Thu    2068.0
Fri    2010.0
Sat    2976.0
Sun    2370.0

Caution: under 10 posts per group is a hint, not proof. Re-check monthly.


### Example · `session-07_ab_test.py`

In [4]:
"""
Session 7 · Example 2 — Did headline B really win? A two-proportion z-test.

AI can write ten headline variants in seconds, but "B got more clicks" might
be luck. The z-test compares two click-through rates:
  p = pooled rate, SE = sqrt(p(1-p)(1/nA + 1/nB)),  z = (pB - pA) / SE
|z| >= 1.96 means the difference is significant at the 95% level.
Only the standard library is needed (math.erf gives the normal curve).

Run:  python session-07_ab_test.py
"""
from math import erf, sqrt


def ab_test(clicks_a, n_a, clicks_b, n_b):
    pa, pb = clicks_a / n_a, clicks_b / n_b
    pooled = (clicks_a + clicks_b) / (n_a + n_b)
    se = sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z = (pb - pa) / se
    p_value = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
    return pa, pb, z, p_value


tests = [
    ("Small test", 12, 400, 19, 400),
    ("Same rates, 10x traffic", 120, 4000, 190, 4000),
]
print("A: 'Admissions open for 2027'   B: 'Your CS career starts here: apply by 15 Jan'")
for name, ca, na, cb, nb in tests:
    pa, pb, z, p = ab_test(ca, na, cb, nb)
    verdict = "B wins (significant)" if p < 0.05 else "not significant: keep testing"
    print(f"{name:24} CTR A={pa:.2%} B={pb:.2%}  lift={(pb/pa-1):+.0%}  z={z:.2f}  p={p:.4f}  -> {verdict}")

A: 'Admissions open for 2027'   B: 'Your CS career starts here: apply by 15 Jan'
Small test               CTR A=3.00% B=4.75%  lift=+58%  z=1.28  p=0.1997  -> not significant: keep testing
Same rates, 10x traffic  CTR A=3.00% B=4.75%  lift=+58%  z=4.06  p=0.0001  -> B wins (significant)


### Example · `session-07_project_content_audit.py`

In [5]:
"""
Session 7 · Mini project (full solution) — A monthly content audit.

Real-life problem: the department's social media lead asks: "What worked
in September, what didn't, and what should we change in October?"
This script answers with data, then builds a prompt so an AI assistant can
turn the numbers into a short written report. Numbers come from code, the
WORDS come from AI, and a human checks both.

Steps:
  1. load and clean the export          4. compare formats and topics
  2. compute KPIs per post              5. chart ER by format (PNG)
  3. top 3 / bottom 3 posts             6. rule-based recommendations + AI prompt

Run:  python session-07_make_data.py  (once)
      python session-07_project_content_audit.py
Creates: session-07_er_by_format.png
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# 1) load and clean
df = pd.read_csv("session-07_posts.csv")
before = len(df)
df = df[(df.reach > 0) & (df.impressions >= df.reach)].copy()   # impossible rows removed
print(f"Loaded {before} posts, {before - len(df)} removed as invalid")

# 2) KPIs
df["ER"] = 100 * df.engagements / df.reach
df["CTR"] = 100 * df.clicks / df.impressions

# 3) best and worst
cols = ["post_id", "format", "topic", "hour", "reach", "ER"]
print("\nTop 3 posts by engagement rate:")
print(df.nlargest(3, "ER")[cols].round(2).to_string(index=False))
print("\nBottom 3 posts by engagement rate:")
print(df.nsmallest(3, "ER")[cols].round(2).to_string(index=False))

# 4) format and topic comparison (rates from totals)
def rate_table(key):
    g = df.groupby(key)[["engagements", "reach", "clicks", "impressions", "signups"]].sum()
    g["ER %"] = 100 * g.engagements / g.reach
    g["CTR %"] = 100 * g.clicks / g.impressions
    g["posts"] = df.groupby(key).size()
    return g[["posts", "ER %", "CTR %", "signups"]].round(2).sort_values("ER %", ascending=False)

fmt = rate_table("format")
topic = rate_table("topic")
print("\nBy format:\n" + fmt.to_string())
print("\nBy topic:\n" + topic.to_string())

# 5) chart
plt.figure(figsize=(6, 3.2))
plt.bar(fmt.index, fmt["ER %"], color=["#FF6B1A", "#FCD34D", "#A78BFA", "#F43F5E"][:len(fmt)])
avg = 100 * df.engagements.sum() / df.reach.sum()
plt.axhline(avg, color="black", linestyle="--", linewidth=1, label=f"overall {avg:.1f}%")
plt.ylabel("Engagement rate %")
plt.title("September: engagement rate by format")
plt.legend()
plt.tight_layout()
plt.savefig("session-07_er_by_format.png", dpi=80)
print("\nSaved -> session-07_er_by_format.png")

# 6) recommendations from rules, then a prompt for the written report
best_fmt, worst_fmt = fmt.index[0], fmt.index[-1]
hour_er = df.groupby("hour").apply(lambda g: 100 * g.engagements.sum() / g.reach.sum())
recs = [
    f"Shift effort toward {best_fmt} posts ({fmt.loc[best_fmt, 'ER %']:.1f}% ER) and away from "
    f"{worst_fmt} ({fmt.loc[worst_fmt, 'ER %']:.1f}%).",
    f"Schedule posts at {hour_er.idxmax()}:00, the best-performing hour ({hour_er.max():.1f}% ER).",
    f"Topic '{topic['signups'].idxmax()}' drove the most sign-ups ({int(topic['signups'].max())}): "
    "add one more post on it per week.",
    "A/B test two headlines on the next admissions post before scaling.",
]
print("\nRECOMMENDATIONS")
for r in recs:
    print(" -", r)

print("\nPROMPT FOR THE WRITTEN REPORT")
print("You are a social media analyst. Using ONLY the numbers below, write a 150-word report "
      "for the department head: what worked, what did not, three actions for October. "
      "Do not invent any figure.\n" + fmt.to_string() + "\nRecommendations: " + " ".join(recs))

Loaded 60 posts, 0 removed as invalid

Top 3 posts by engagement rate:
post_id format        topic  hour  reach   ER
    P42   reel     research    17   5031 6.28
    P12   reel student life    13   2276 6.24
    P55   reel   admissions     9   2715 5.86

Bottom 3 posts by engagement rate:
post_id format        topic  hour  reach   ER
    P46   text student life     9   1007 1.49
    P13   text         tips    20   1579 1.52
    P24   text         tips    20   1111 1.53

By format:
          posts  ER %  CTR %  signups
format                               
reel         21  4.56   0.54       24
carousel     15  4.23   0.66       13
image         9  3.16   0.70        3
text         15  1.82   0.60        1

By topic:
              posts  ER %  CTR %  signups
topic                                    
admissions        9  4.43   0.50        8
student life     10  4.15   0.62        7
research         12  4.12   0.52        8
tips             21  3.78   0.69       17
events            8  3